# diffsvg: full tutorial

diffsvg finds **differential spatially variable genes**: genes whose expression changes differently with the distance to an anchor in two conditions. This tutorial uses legnini et al.: 17 slides of neural organoid-like cultures, 6 Ctrl and 11 exposed to SHH, 88 genes. The anchor is the SHH+ cells.

1. **Quickstart**: the fastest route from data to a volcano plot, on one model for the whole dataset.
2. **Grid search**: how much the ranking depends on the settings, and which settings to keep.
3. **The full model**: those settings, plus the variation between slides.
4. **Clustering**: two ways of grouping the genes by their curves.

The fits run in parallel, one model per gene. With 4 workers parts 1 and 3 take a few minutes each, and part 2 about 40 minutes (10 refits of 88 genes).

In [ ]:
# One BLAS thread per process: the fits run in worker processes, and oversubscribed BLAS threads slow them down a lot.
# This has to happen before numpy is imported, so restart the kernel if numpy is already loaded.
import os

for var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[var] = "1"

import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq

import diffsvg as dvg

N_WORKERS = int(os.environ.get("DIFFSVG_WORKERS", 4))
LAYER = "log1p_norm"
TAU = 0.1
timings = {}

## Data and anchor

`var_by_distance` gives every cell its distance to the closest anchor cell of its own slide, scaled to 0-1 per slide, and collects what the model needs in `adata.obsm["design_matrix"]`. Here, the anchor is the top percentile of SHH expression. Anchor cells themselves have no distance, and `slide1_A2-2` has no anchor cells, so it drops out of the fits.

In [ ]:
adata = sc.read_h5ad(os.environ.get("DIFFSVG_DATA", "/home/lehnerl/Arbeit/data/legnini23.h5ad"))

shh = adata[:, "SHH"].X
shh = (shh.toarray() if hasattr(shh, "toarray") else np.asarray(shh)).ravel()
adata.obs["SHH+"] = (shh > np.percentile(shh, 99)).astype(str)

sq.tl.var_by_distance(adata, cluster_key="SHH+", groups="True", library_key="sample", covariates="condition")
design = adata.obsm["design_matrix"]
design.head()

In [ ]:
EXAMPLE_SLIDES = ["slide4_B2-3", "slide1_A2-1"]

fig, axes = plt.subplots(1, 2, figsize=(9, 4), constrained_layout=True)
for ax, slide in zip(axes, EXAMPLE_SLIDES, strict=True):
    in_slide = (design["sample"] == slide).to_numpy()
    anchor = in_slide & design["True"].isna().to_numpy()
    others = in_slide & ~anchor
    xy = adata.obsm["spatial"]
    points = ax.scatter(*xy[others].T, c=design["True"].to_numpy()[others], cmap="viridis", vmin=0, vmax=1, s=4, linewidth=0)
    ax.scatter(*xy[anchor].T, color="red", s=8, linewidth=0, label="SHH+ (anchor)")
    ax.set_title(f"{slide} ({design.loc[in_slide, 'condition'].iloc[0]})", fontsize=10)
    ax.set_aspect("equal")
    ax.set_axis_off()
    ax.legend(frameon=False, fontsize=8, loc="lower right")
fig.colorbar(points, ax=axes, shrink=0.7, label="Distance to the closest anchor cell");

## 1. Quickstart

Three calls and a plot:

* `pp.bin_obs` averages each gene in bins along the distance, per condition. `window_radius=None` makes the bins contiguous, so each cell counts once.
* `tl.fit_model` fits one spline per gene and condition, and tests whether the two curves differ by more than a shift in level (`test="parallel"`). `tau` is the smallest shape difference that counts.
* `pl.plot_volcano` shows the effect (`slope_diff`, how the gap between the curves changes with distance) against the significance.

### Why ADVI

The fastest route on this data is variational inference on contiguous bins, not NUTS. Measured with 4 workers on 24 genes, twice each, in seconds per gene:

| Fit | s / gene |
|---|---|
| Curve test, contiguous bins, **ADVI** | **1.1** |
| Slide model, per-slide contiguous bins, ADVI | 2.0 |
| Wald, sliding windows, NUTS | 2.5 |
| Curve test, contiguous bins, NUTS | 3.0 |

ADVI and NUTS rank the genes the same here (Spearman 1.00 on the statistic). ADVI needs `window_radius=None`: on the overlapping windows of the `bin_obs` default, a sparse gene can make the optimiser fail with NaN, which stops the whole fit.

In [ ]:
t0 = time.perf_counter()

dvg.pp.bin_obs(adata, anchor="True", condition="condition", layer=LAYER, window_radius=None)
setup = dvg.tl.setup_model(adata)
dvg.tl.fit_model(adata, setup, n_workers=N_WORKERS, inference="advi", test="parallel", tau=TAU)

timings["1. Quickstart"] = time.perf_counter() - t0
print(f"{timings['1. Quickstart']:.0f} s")

quick = adata.varm["dsvg_stats"].copy()
dvg.pl.plot_volcano(quick, n_label=15, figsize=(7, 5.5));

The curves are compared as condition 1 minus condition 0, with the labels in sorted order, so here SHH minus Ctrl. Blue genes have a gap that grows with the distance from the anchor, red genes one that shrinks.

`plot_prediction_curve` shows what is behind a point of the volcano. Bands are posterior-predictive, so they hold the noise of a bin as well as the uncertainty of the curve: judge the difference by the test and the mean curves.

In [ ]:
quick_top = list(quick.sort_values(["parallel_pval", "parallel_stat"], ascending=[True, False]).index[:4])
dvg.pl.plot_prediction_curve(adata, gene=quick_top);

**This model is too confident on data with several slides.** It pools the cells of a condition, so a difference between two slides looks like a difference between the conditions, and with 40,000 cells it is significant. On a split of the SHH slides into two groups, where no true difference exists, a model like this flags most of the genes. Use it to look around. Part 3 is the one to report.

## 2. Grid search

The model has settings that nothing in the data fixes directly, chiefly the flexibility of the splines (`n_bsplines`) and the number of bins (`n_bins`). `check_stability` refits the whole panel for every combination and compares the rankings. It also fits the baseline a second time with another seed: that is the **noise floor**, how much the ranking moves by chance alone.

The fits here are NUTS on pooled contiguous bins with half the usual draws, since a ranking needs less precision than a final fit. `check_stability` does not offer the slide model, so this tells us how much the settings matter for the spline and not for the slide effects. `rank_by="parallel_stat"` ranks by the statistic of the curve test, which unlike the p-value does not saturate at 0 or 1.

**What "optimal" means here.** There is no ground truth, so no setting can be shown to be the most accurate. What can be measured is whether more flexibility still changes the answer. So the baseline is the richest setting, and we keep the *smallest* setting whose ranking cannot be told apart from it, that is, whose agreement with it is at least the noise floor.

In [ ]:
GRID = {"n_bsplines": [6, 10, 14], "n_bins": [25, 50, 100], "window_radius": [None], "layer": [LAYER]}
RICHEST = {"n_bsplines": 14, "n_bins": 100, "window_radius": None, "layer": LAYER}

t0 = time.perf_counter()
stability = dvg.tl.check_stability(
    adata, anchor="True", condition="condition", grid=GRID, baseline=RICHEST, genes=None,
    rank_by="parallel_stat", top_k=20, n_seeds=2, n_workers=N_WORKERS,
    sample_kwargs={"draws": 500, "tune": 500}, test="parallel", tau=TAU,
)  # fmt: skip
timings["2. Grid search"] = time.perf_counter() - t0
print(f"{timings['2. Grid search'] / 60:.1f} min, noise floor {stability.noise_floor}")

The heatmap compares every run with every other. Runs on the same branch of the dendrogram rank the genes the same way, and the red line on the colour bar is the noise floor.

In [ ]:
dvg.pl.plot_stability(stability, kind="heatmap", metric="spearman");

The same comparison against the baseline, with one setting changed at a time, shows which setting moves the ranking.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
dvg.pl.plot_stability(stability, kind="sensitivity", metric="spearman", ax=axes[0])
dvg.pl.plot_stability(stability, kind="sensitivity", metric="top_overlap", ax=axes[1])
fig.tight_layout();

In [ ]:
runs = stability.configs[~stability.configs.index.str.contains("reseed")][["n_bsplines", "n_bins"]].astype(int)
runs["spearman"] = stability.similarity.loc[stability.baseline, runs.index]
runs["top_overlap"] = stability.top_overlap.loc[stability.baseline, runs.index]
runs["within noise"] = (runs["spearman"] >= stability.noise_floor["spearman"]) & (
    runs["top_overlap"] >= stability.noise_floor["top_overlap"]
)

best = runs[runs["within noise"]].sort_values(["n_bsplines", "n_bins"]).iloc[0]
N_BSPLINES, N_BINS = int(best["n_bsplines"]), int(best["n_bins"])
print(f"Kept: n_bsplines = {N_BSPLINES}, n_bins = {N_BINS}")
runs.sort_values(["n_bsplines", "n_bins"]).round(3)

## 3. The full model

The chosen settings, now with slides: `sample="sample"` adds a random level and a random curve per slide. That is what the over-confidence of the quickstart was missing, because the conditions are now compared against the variation between slides rather than between cells.

* **Empirical Bayes** (`slide_scales="empirical_bayes"`): the two scales of the slide effects are estimated per gene, as the mode of their marginal posterior, then held fixed while ADVI fits the rest. Letting ADVI fit them too does not work.
* **F reference:** the p-value uses `n_slides - 2` denominator degrees of freedom, because the covariance now holds scale estimates from few slides.
* **Bins or cells:** contiguous bins per slide are weighted by their cell counts, so they have the same likelihood as the cells. On this data they gave nearly the same statistic as a fit to every cell (HHIP 139.5 vs. 138.6, FOXA2 152.7 vs. 151.4, GLI1 117.2 vs. 114.3, PAX7 92.3 vs. 93.1) in 3 instead of 34 minutes with 4 workers. For the fit to cells, set `USE_BINNED = False`.

In [ ]:
USE_BINNED = True

t0 = time.perf_counter()
dvg.pp.bin_obs(
    adata, anchor="True", condition="condition", layer=LAYER, n_bsplines=N_BSPLINES, n_bins=N_BINS,
    window_radius=None, sample="sample",
)  # fmt: skip
dvg.tl.fit_model(
    adata, dvg.tl.setup_model(adata), n_workers=N_WORKERS, use_binned=USE_BINNED, inference="advi",
    slide_scales="empirical_bayes", test="parallel", tau=TAU,
)  # fmt: skip
timings["3. Full model"] = time.perf_counter() - t0
print(f"{timings['3. Full model'] / 60:.1f} min")

stats = adata.varm["dsvg_stats"].copy()

### Did the approximation work?

`vi_khat` and `vi_ess` compare each gene's approximation with its true posterior. With slide effects, trust a gene with `vi_khat < 0.7` and `vi_ess > 0.1`. `fit_model` warns when genes fail.

In [ ]:
stats["vi_trusted"] = (stats["vi_khat"] < 0.7) & (stats["vi_ess"] > 0.1)
print(f"{stats['vi_trusted'].sum()} of {len(stats)} genes pass the variational diagnostics")
stats.loc[~stats["vi_trusted"], ["vi_khat", "vi_ess", "parallel_pval"]]

### Ranked genes

In [ ]:
ranked = stats.sort_values(["parallel_pval", "parallel_stat"], ascending=[True, False])
ranked[
    ["shape_diff", "slope_diff", "slope_diff_ci_low", "slope_diff_ci_high", "parallel_stat", "parallel_pval",
     "parallel_pval_adj", "integral", "sample_level_sd", "sample_curve_sd", "vi_trusted"]
].head(20)  # fmt: skip

### Volcano and p-value histogram

With `tau > 0`, genes whose shape difference is clearly below `tau` get p-values near 1, so the histogram has a second peak there by design.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
dvg.pl.plot_pvalue_histogram(stats, ax=axes[0])
dvg.pl.plot_volcano(stats, ax=axes[1], n_label=15)
fig.tight_layout();

### What modelling the slides changed

The quickstart against the full model. Most genes the quickstart flags are no longer significant once the variation between slides is taken into account.

In [ ]:
display(
    pd.DataFrame(
        {
            "Quickstart": {"p < 0.05": (quick["parallel_pval"] < 0.05).sum(), "BH < 0.05": (quick["parallel_pval_adj"] < 0.05).sum()},
            "Full model": {"p < 0.05": (stats["parallel_pval"] < 0.05).sum(), "BH < 0.05": (stats["parallel_pval_adj"] < 0.05).sum()},
        }
    )
)
print("Spearman of the statistics:", round(quick["parallel_stat"].corr(stats["parallel_stat"], method="spearman"), 2))

FLAGGED = list(ranked.index[ranked["parallel_pval_adj"] < 0.05])
print("Flagged by the full model (BH < 0.05):", ", ".join(FLAGGED))
dvg.pl.plot_prediction_curve(adata, gene=list(ranked.index[:8]));

## 4. Clustering

`tl.cluster_curves` runs k-means on the posterior means of the spline coefficients, so genes group by the shape of their curves. There are two ways to do this, and they answer different questions:

* **Combined** (`by="average"`): the mean of the two condition curves. Which patterns along the distance exist in this data?
* **Interaction** (`by="interaction"`): the difference between the conditions. In which ways do the conditions differ?

### Choosing `k`

The silhouette (higher is better), for both. It tends to favour small `k`, where a large flat cluster stands against a few strong patterns, so it is a guide and does not settle the choice.

In [ ]:
silhouettes = {}
for by in ("average", "interaction"):
    for k in range(2, 9):
        dvg.tl.cluster_curves(adata, by=by, n_clusters=k, key_added="scan")
        silhouettes[by, k] = adata.uns["scan"]["silhouette"]
del adata.var["scan"], adata.uns["scan"]
pd.Series(silhouettes).unstack(0).round(3).rename_axis("k")

In [ ]:
N_CLUSTERS = 5


def cluster_table(key):
    """One row per cluster: its size, how many of its genes the test flags, and the genes."""
    labels = adata.var[key].dropna()
    return pd.DataFrame(
        {
            cluster: {
                "n": len(genes),
                "p < 0.05": int((stats.loc[genes, "parallel_pval"] < 0.05).sum()),
                "BH < 0.05": int((stats.loc[genes, "parallel_pval_adj"] < 0.05).sum()),
                "median shape_diff": round(float(stats.loc[genes, "shape_diff"].median()), 2),
                "genes": ", ".join(genes),
            }
            for cluster, genes in ((c, list(labels.index[labels == c])) for c in labels.cat.categories)
        }
    ).T

### Combined: the curve averaged over both conditions

`beta + beta_1 / 2` are the coefficients of the mean of the two condition curves, weighting both conditions equally. Each gene is centred, so its level plays no part.

In [ ]:
dvg.tl.cluster_curves(adata, by="average", n_clusters=N_CLUSTERS)
display(cluster_table("dsvg_cluster_average"))
dvg.pl.plot_cluster_curves(adata, "dsvg_cluster_average");

### Interaction: how the conditions differ

`beta_1` is the shape of the difference curve, SHH minus Ctrl. The plot shows those difference curves, since they are what the clustering compared. Amplitude counts here, so genes without a difference collect in a flat cluster.

These are the same coefficients the test is built on, so finding the flagged genes in the patterned clusters describes them and does not confirm them.

In [ ]:
dvg.tl.cluster_curves(adata, by="interaction", n_clusters=N_CLUSTERS)
display(cluster_table("dsvg_cluster_interaction"))
dvg.pl.plot_cluster_curves(adata, "dsvg_cluster_interaction");

`curves="conditions"` draws the same clusters split by condition: one row per cluster, Ctrl left, SHH right, in a shared y-range and with the same gene colours. Each gene is centred over both conditions together, so the gap between them stays visible. Here, the clusters that hold flagged genes:

In [ ]:
labels = adata.var["dsvg_cluster_interaction"]
with_flagged = sorted(set(labels[FLAGGED])) or None
dvg.pl.plot_cluster_curves(adata, "dsvg_cluster_interaction", curves="conditions", clusters=with_flagged);

## Runtime

In [ ]:
pd.Series(timings, name="seconds").round(0).to_frame()